# CSV a JSON con pandas

Este cuaderno lee el fichero CSV `notas-de-corte-universides-madrid-25-26.csv` y genera un fichero JSON con **exactamente** estas columnas:

- `Nombre de la Universidad`
- `Rama de estudios`
- `Nombre de la titulación`
- `Nota de corte ordinaria del grupo 1`

El proceso incluye:

- comprobación de existencia del fichero
- lectura del CSV con separador `;`
- manejo de codificación UTF-8 y UTF-8 con BOM
- validación de columnas esperadas
- vista previa de los datos
- exportación a JSON UTF-8 como lista de objetos


In [3]:
# pip install pandas

In [4]:
# Importamos las librerías necesarias.
# pandas se usará para leer y manipular el CSV.
# json se usará para escribir el fichero de salida.
# pathlib facilita el trabajo con rutas de archivos.

import json
from pathlib import Path

import pandas as pd


In [5]:
# Definimos las rutas de entrada y salida.
# Por defecto se asume que el CSV está en el mismo directorio
# desde el que se ejecuta el notebook.
# Si fuera necesario, modifica solo INPUT_CSV.

INPUT_CSV = Path("notas-de-corte-universides-madrid-25-26.csv")
OUTPUT_JSON = Path("notas-de-corte-universides-madrid-25-26.json")

# Definimos también la lista exacta de columnas esperadas.
EXPECTED_COLUMNS = [
    "Nombre de la Universidad",
    "Rama de estudios",
    "Nombre de la titulación",
    "Nota de corte ordinaria del grupo 1",
]

print(f"Archivo CSV esperado: {INPUT_CSV.resolve()}")
print(f"Archivo JSON de salida: {OUTPUT_JSON.resolve()}")


Archivo CSV esperado: /home/adolfo/Cursos/2026/curso-practico-ia-generativa-2026/notas-de-corte-universides-madrid-25-26/notas-de-corte-universides-madrid-25-26.csv
Archivo JSON de salida: /home/adolfo/Cursos/2026/curso-practico-ia-generativa-2026/notas-de-corte-universides-madrid-25-26/notas-de-corte-universides-madrid-25-26.json


In [6]:
# Validamos que el archivo de entrada exista antes de intentar leerlo.
if not INPUT_CSV.exists():
    raise FileNotFoundError(
        f"No se ha encontrado el archivo CSV: {INPUT_CSV.resolve()}"
    )

print("✅ El archivo CSV existe.")


✅ El archivo CSV existe.


In [7]:
# Función de lectura robusta del CSV.
#
# Aspectos importantes:
# - sep=';' porque el archivo usa punto y coma como separador.
# - dtype=str para conservar todos los valores como texto.
#   Esto es especialmente útil para mantener la nota exactamente
#   como aparece en el CSV, por ejemplo "5,000".
# - keep_default_na=False evita que cadenas vacías se conviertan
#   automáticamente en NaN.
# - Se intenta primero con 'utf-8-sig', que soporta tanto UTF-8
#   normal como UTF-8 con BOM.

def read_csv_robust(csv_path: Path) -> pd.DataFrame:
    encodings_to_try = ["utf-8-sig", "utf-8"]
    last_error = None

    for encoding in encodings_to_try:
        try:
            df = pd.read_csv(
                csv_path,
                sep=";",
                encoding=encoding,
                dtype=str,
                keep_default_na=False,
            )
            print(f"✅ CSV leído correctamente con codificación: {encoding}")
            return df
        except Exception as exc:
            last_error = exc
            print(f"⚠️ No se pudo leer con {encoding}: {exc}")

    raise RuntimeError(
        "No fue posible leer el CSV con las codificaciones probadas."
    ) from last_error


df = read_csv_robust(INPUT_CSV)


✅ CSV leído correctamente con codificación: utf-8-sig


In [8]:
# Validamos que todas las columnas esperadas estén presentes.
current_columns = df.columns.tolist()
missing_columns = [col for col in EXPECTED_COLUMNS if col not in current_columns]

print("Columnas detectadas en el CSV:")
for col in current_columns:
    print(f" - {col}")

if missing_columns:
    raise ValueError(
        "Faltan columnas obligatorias en el CSV: "
        + ", ".join(missing_columns)
    )

print("\n✅ Todas las columnas esperadas están presentes.")


Columnas detectadas en el CSV:
 - Nombre de la Universidad
 - Rama de estudios
 - Nombre de la titulación
 - Nota de corte ordinaria del grupo 1

✅ Todas las columnas esperadas están presentes.


In [9]:
# Seleccionamos únicamente las cuatro columnas requeridas,
# en el orden exacto solicitado.
df_selected = df[EXPECTED_COLUMNS].copy()

# Mostramos información básica para validación.
print(f"Filas leídas: {len(df_selected)}")
print("\nVista previa de los primeros registros:")
display(df_selected.head(10))


Filas leídas: 594

Vista previa de los primeros registros:


,Nombre de la Universidad,Rama de estudios,Nombre de la titulación,Nota de corte ordinaria del grupo 1
0,Universidad Autónoma de Madrid,Artes y Humanidades,Arqueología,"5,000"
1,Universidad Autónoma de Madrid,Artes y Humanidades,Estudios Clásicos y de la Antigüedad,"5,000"
2,Universidad Autónoma de Madrid,Artes y Humanidades,Estudios Hispánicos: Lengua Española y sus Lit...,"5,000"
3,Universidad Autónoma de Madrid,Artes y Humanidades,Estudios Ingleses,"5,000"
4,Universidad Autónoma de Madrid,Artes y Humanidades,Estudios de Asia y África (Chino),"5,000"
5,Universidad Autónoma de Madrid,Artes y Humanidades,Estudios de Asia y África (Japonés),"5,000"
6,Universidad Autónoma de Madrid,Artes y Humanidades,Estudios de Asia y África (Árabe),"5,000"
7,Universidad Autónoma de Madrid,Artes y Humanidades,Filosofía,"5,000"
8,Universidad Autónoma de Madrid,Artes y Humanidades,Historia,"5,000"
9,Universidad Autónoma de Madrid,Artes y Humanidades,Historia del Arte,"5,000"


In [10]:
# Convertimos el DataFrame en una lista de diccionarios.
# Cada diccionario representará una fila del CSV y las claves
# serán exactamente los nombres de columna requeridos.
records = df_selected.to_dict(orient="records")

# Guardamos el JSON en UTF-8.
# ensure_ascii=False permite conservar correctamente acentos y caracteres Unicode.
# indent=2 hace que el archivo sea más legible.
with OUTPUT_JSON.open("w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=2)

print(f"✅ JSON generado correctamente con {len(records)} registros.")


✅ JSON generado correctamente con 594 registros.


In [11]:
# Mostramos una pequeña vista previa del JSON generado
# leyendo los primeros elementos desde la variable en memoria.
records[:3]


[{'Nombre de la Universidad': 'Universidad Autónoma de Madrid',
  'Rama de estudios': 'Artes y Humanidades',
  'Nombre de la titulación': 'Arqueología',
  'Nota de corte ordinaria del grupo 1': '5,000'},
 {'Nombre de la Universidad': 'Universidad Autónoma de Madrid',
  'Rama de estudios': 'Artes y Humanidades',
  'Nombre de la titulación': 'Estudios Clásicos y de la Antigüedad',
  'Nota de corte ordinaria del grupo 1': '5,000'},
 {'Nombre de la Universidad': 'Universidad Autónoma de Madrid',
  'Rama de estudios': 'Artes y Humanidades',
  'Nombre de la titulación': 'Estudios Hispánicos: Lengua Española y sus Literaturas',
  'Nota de corte ordinaria del grupo 1': '5,000'}]

In [12]:
# Confirmación final de la ruta del archivo generado.
if OUTPUT_JSON.exists():
    print("✅ Proceso completado.")
    print(f"Ruta del JSON generado: {OUTPUT_JSON.resolve()}")
else:
    raise FileNotFoundError(
        f"No se ha generado el JSON esperado: {OUTPUT_JSON.resolve()}"
    )


✅ Proceso completado.
Ruta del JSON generado: /home/adolfo/Cursos/2026/curso-practico-ia-generativa-2026/notas-de-corte-universides-madrid-25-26/notas-de-corte-universides-madrid-25-26.json
